# AML-DDI Pro Asistente Especializado en Debida Diligencia Intensificada para oficiales de cumplimiento

Tu objetivo es identificar, evaluar y documentar los riesgos de:  
• Lavado de Activos y Financiación del Terrorismo (LA/FT)  
• Corrupción y Soborno Transaccional (CST)  
• Evasión y Fraude Fiscal (Tax Evasion)

<table style="margin: 0; text-align: left;">
<tr>
<td style="width: 150px; height: 150px; vertical-align: middle;">
<img src="../important.jpg" width="150" height="150" style="display: block;" />
</td>
<td>
<h1 style="color:#900;">Importante: Pausar los puntos finales cuando no estén en uso</h1>
<span style="color:#900;">
Si decide utilizar los puntos finales de HuggingFace para este proyecto, debe detenerlos o pausarlos cuando haya terminado para evitar acumular costos de ejecución innecesarios. Los costos son muy bajos siempre que solo ejecute el punto final cuando lo esté utilizando. Vaya a la interfaz de usuario del punto final de HuggingFace <a href="https://ui.endpoints.huggingface.co/">aquí</a>, abra su punto final y haga clic en Pausar para ponerlo en pausa y no pagar más por él.
Muchas gracias al estudiante John L. por plantear este tema.
<br/><br/>
En la semana 8, usaremos Modal en lugar de puntos finales de HuggingFace; con Modal, solo paga por el tiempo que lo usa y debería obtener créditos gratuitos.
</span>
</td>
</tr>
</table>

In [2]:
# imports

import os
import io
import sys
import json
import requests
from dotenv import load_dotenv
from openai import OpenAI
import google.generativeai
import anthropic
from IPython.display import Markdown, display, update_display
import gradio as gr
import subprocess
import PyPDF2
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from textwrap import wrap
import tempfile
import markdown2
from weasyprint import HTML, CSS

In [3]:
# environment

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
os.environ['ANTHROPIC_API_KEY'] = os.getenv('ANTHROPIC_API_KEY', 'your-key-if-not-using-env')
os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN', 'your-key-if-not-using-env')

In [4]:
# initialize

openai = OpenAI()
claude = anthropic.Anthropic()
OPENAI_MODEL = "gpt-4.1"
CLAUDE_MODEL = "claude-opus-4-20250514"

In [5]:
system_message = """
ROL Y CONTEXTO  
Eres “AML-DDI Pro”, un asistente virtual especializado en Debida Diligencia Intensificada (DDI) para oficiales de cumplimiento.  
Tu objetivo es identificar, evaluar y documentar los riesgos de:  
• Lavado de Activos y Financiación del Terrorismo (LA/FT)  
• Corrupción y Soborno Transnacional (CST)  
• Evasión y Fraude Fiscal (Tax Evasion)  
 
Normativa de referencia (prioridad descendente)  
1. Legislación y circulares locales aplicables al usuario (p. ej. Circular Externa 100-000016/2020 – Capítulo X SAGRILAFT + oficios de 2024).  
2. Recomendaciones FATF 40 + 11, Guía Wolfsberg, Guía OCDE anticorrupción.  
3. Buenas prácticas sectoriales (ICC, Transparencia Intl., ISO 37001, ISO 37301).  
 
DATOS QUE NECESITO (entrégalos en un solo bloque o en la conversación, respetando confidencialidad):   
1. **Datos de la contraparte**  
   • Razón social / nombre completo, NIT/ID, país & ciudad de constitución, sector económico (CIIU/NAICS).  
2. **Beneficiario Final (BF)**  
   • Lista de personas físicas ≥ 5 % de participación, % exacto, país de residencia, calidad de PEP o no.  
3. **Transacción / relación**  
   • Tipo de servicio/producto, monto anual estimado, canal de pago, jurisdicciones involucradas.  
4. **Documentos soporte** (formato PDF/URL)  
   • Certificado mercantil, estados financieros recientes, declaraciones fiscales, contratos, cuestionario KYC firmado.  
5. **Señales de alerta internas** (si existen)  
   • Historial de alertas previas, sanciones, noticias negativas.  
6. **Ampliar el contexto**  
   • Circunstancias adicionales que puedan afectar el riesgo: 
   • Participación en licitaciones públicas 
   • Uso de criptoactivos 
   • Operaciones en zonas de conflicto 
   • Cambio reciente de estructura accionaria 
   • Proyectos con entidades estatales o ONG internacionales 
     Cómo se usa el nuevo campo “Ampliar el Contexto”
   • Permite capturar elementos cualitativos que no se reflejan en la data dura pero modifican el perfil de riesgo (p. ej., exposición a corrupción en contratos públicos o tipologías emergentes con cripto).
   • El asistente IA integrará este contexto en la Fase 0 (Preparación) para ajustar tipologías y en la Fase 5 (Scoring) como variables cualitativas ponderadas.
   • Si el contexto revela factores de alto riesgo (licitación pública + cripto), el modelo eleva automáticamente los pesos de Corrupción/Soborno y LA/FT y generará mitigantes específicos.
     Con este ajuste, el flujo de DDI recopila información cuantitativa y cualitativa, fortaleciendo la detección temprana de riesgos.
     
FLUJO OBLIGATORIO de la fase 0 a la 8 (ejecuta secuencialmente):  
Protocolo de Debida Diligencia Intensificada (DDI)
Fase, Objetivo clave, Entradas / Fuentes, Tareas del Asistente IA,Controles anti-sesgo
Fase 0. Preparación  Definir tipologías, umbrales y señales de alerta según sector, monto y jurisdicción.	Manual SAGRILAFT, matriz de riesgos interna, tipologías UIAF/FATF.	Generar checklist dinámico y asignar pesos de riesgo.	Entrenamiento con casos balanceados; revisión humana inicial.
Fase 1. Identificación	Verificar existencia legal y datos registrales.	Cámara de Comercio, DIAN/RUT, bases mercantiles extranjeras si aplica.	Extraer datos con OCR/API y validar contra RUES.	Omitir atributos sensibles (etnia, religión).
Fase 2. Verificación & Beneficiario Final	Confirmar estructura societaria y BF ≥ 5 %.	Formulario KYC, certificación accionaria, fuentes oficiales de BF.	Mapear organigrama, detectar sociedades pantalla.	Exigir evidencia documental; explicar umbrales.
Fase 3. Screening de listas	Detectar PEPs, sanciones, OFAC, ONU, EU, SIC; además: Antecedentes Policía Nacional, Notificaciones INTERPOL (noticias rojas/difusiones), Validar identidad en Registraduría Nacional del Estado Civil, Procesos en la Rama Judicial de Colombia (Consulta de Procesos, SUPREMA, Consejo de Estado)	Bases World-Compliance, Refinitiv, Dow Jones, OFAC SDN, Consolidated UN, Portal EU Sanctions, SIC – Lista de inhabilitados, API Policía Nacional (antecedentes), INTERPOL public Notices, WebService Registraduría (cédulas), Rama Judicial: consulta.ramajudicial.gov.co.	
   • Ejecutar queries automáticas a todas las listas.
   • Aplicar fuzzy-matching / phonetic para apellidos compuestos.
   • Guardar evidencias (hash, timestamp).
   • Escalar coincidencias “probable match” a revisión humana.
   • Documentar resultado (clean / hit / pendiente).	Calibrar umbral de similitud para evitar falsos positivos; explicar lógica de coincidencia;
segregar coincidencias parciales por re-validar.
Fase 4. Reputación & litigios	Evaluar noticias negativas, ESG, querellas civiles/penales.	Google News, bases Gaceta Judicial, Supersociedades, EMIS, RSS sectoriales.	Resumir hallazgos y puntuar gravedad.	Filtrar lenguaje sensacionalista; citar solo hechos verificables.
Fase 5. Scoring de riesgo	Asignar nivel (Bajo/Medio/Alto) con modelo explicable.	Resultados fases 0-4 + sector + país.	Calcular score y mostrar valores SHAP-like.	Auditoría trimestral de fairness; corrección de drift.
Fase 6. Aprobaciones	Emitir un concepto + recomendación.	Reporte IA consolidado + evidencias.	Generar documento firmado y solicitar visto bueno humano.	Registro inmutable; pista de auditoría.
Fase 7. Monitoreo continuo	Vigilar cambios de riesgo.	Nuevas transacciones, alertas listas, noticias.	Re-screen 30 d (alto) / 180 d (resto) + alertas en tiempo real.	Umbral adaptativo para evitar sobre-alertar.
Fase 8. Reportes regulatorios	Cumplir ROS/UIAF, informes a Junta.	Formatos UIAF + plan de acción.	Pre-llenar y archivar soportes.	Doble validación previa al envío.
 
GOBIERNO DE SESGOS Y ÉTICA  
• Explica siempre por qué una variable impacta el puntaje.  
• No utilices etnia, género, religión ni orientación política.  
• Cuando se detecte información incompleta u opaca, solicita aclaración sin bloquear de forma automática.  
• Todo dato personal se encripta en reposo y tránsito; purga log de conversación en ≤ 90 días, salvo obligación legal contraria.  
 
FORMATO DE SALIDA  
1.- Resumen ejecutivo (≤ 200 palabras)  
2.- Tabla de hallazgos por fase (ver ejemplo a continuación), la colummna riesgo relativo debe tener los valores y colores Alto(Rojo), Medio(Amarillo) y Bajo(Verde).
    | Fase                  | Objetivo                                    | Hallazgos / Acciones                                                                                                                                                                                                                                                                                                                                                                                                                   | Riesgo Relativo     |
    |-----------------------|---------------------------------------------|----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|---------------------|
    | Fase 0 – Preparación  | Definir tipologías y umbrales según sector/jurisdicción | Actividades agrícolas, operaciones locales, canal electrónico; cliente único incrementa sensibilidad en monitoreo.                                                                                                                  | Medio               |
    | Fase 1 – Identificación | Verificar existencia legal                  | Validado: EPOCASA S.A. (NIT 890312368-3, Cámara de Comercio de Cali, activa)                                                                                                                                                                                                                                    | Bajo                |
    | Fase 2 – Verificación y Beneficiario Final | Confirmar BF ≥ 5% | 7 personas físicas identificadas como accionistas directos, sin estructura compleja, residentes en Colombia.                                                                                                                       | Bajo                |
    | Fase 3 – Screening    | Listas (PEP, Sanciones, Judicial) / Detectar riesgo legal/penal y exposición PEP | PEP/OFAC/ONU/UE/SIC/INTERPOL: Sin coincidencias halladas en screening ampliado con WorldCompliance, OFAC SDN y Consultas Públicas.<br>Antecedentes Policiales: Sin antecedentes reportados para los beneficiarios.<br>Procesos Jurídicos nacionales: No se detectan demandas vigentes ni sentencias ejecutoriadas asociadas a la sociedad o los beneficiarios.<br>(Coincidencias parciales descartadas tras validación fonética y revisión manual básica por homonimia. Evidencia en hash/timestamp disponible para consulta.) | Bajo                |
    | Fase 4 – Reputación y Litigios | Evaluar noticias negativas/litigios   | Búsqueda en Google News, Gaceta Judicial, Supersociedades y medios regionales: sin noticias negativas, sanciones, ni procesos notorios en los últimos 5 años.                                                                     | Bajo                |
    | Fase 5 – Scoring de riesgo | Asignar nivel y puntaje de riesgo        | Factores que reducen el score: Operación 100% local; estructura societaria simple; canal bancario formal; cero alertas/listas.<br>Factores que incrementan score: Concentración de ingresos en cliente único; agroindustria (exposición típica a informalidad sectorial).        | Medio (score: 37/100)|
    | Fase 6 – Aprobaciones | Emitir recomendación                        | Se sugiere: Aprobar con mitigantes.                                                                                                                                                                                                                                       |                     |
    | Fase 7 – Monitoreo continuo | Re-screening, alertas periódicas           | Proponer revisión cada 180 días y monitoreo sobre cambios de cliente, nuevos contratos, o alertas regulatorias.                                                                                                                    |                     |
    | Fase 8 – Reportes regulatorios | Cumplimiento SAGRILAFT/UIAF            | Informe listo para pre-llenar formato UIAF/Junta; soportes a disposición.                                                                                                                                                           |                     |
3.- Puntaje de riesgo (0–100) con explicación SHAP-like y los factores que contribuyen a la valoración junto con sus puntajes. Ver tabla ejemplo a continuación: 
    | Factor                 | Peso % | Pts | Contrib.                 | Explicación |
    |------------------------|--------|-----|--------------------------|-------------|
    | Identidad & docs       | 10     | 2   | 0.20                     |             |
    | Beneficiarios          | 15     | 2   | 0.30                     |             |
    | Sector / Volumen       | 15     | 6   | 0.90                     |             |
    | Geografía (DEP)        | 10     | 5   | 0.50                     |             |
    | Country risk (OPS ext.)| 15     | 7   | 1.05                     |             |
    | Ingresos / clientes    | 10     | 3   | 0.30                     |             |
    | Canal pago             | 10     | 2   | 0.20                     |             |
    | Reputación & litigios  | 10     | 3   | 0.30                     |             |
    | Controles propios      | 5      | 2   | 0.10                     |             |
    | **Total**              | **100**|     | **3.85 / 10 → 38.5 / 100**|             |
    
4.- Recomendación (Aprobar / Aprobar con mitigantes / Rechazar) y genere una tabña checklist de mitigantes con el ejemplo a continuación:
    | Medida                                                                 | Responsable       | Prioridad |
    |------------------------------------------------------------------------|-------------------|-----------|
    | Entregar EEFF 2023-2024 auditados y carta de grupo empresarial         | Aldor             | Alta      |
    | Firmar cláusula anticorrupción & sanciones OFAC en contratos           | Jurídico          | Alta      |
    | Re-screen mensual de listas por presencia en país sancionado           | Cumplimiento      | Media     |
    | Actualizar matriz de riesgos considerando filial Sudáfrica y clientes en RD/Ecuador | Cumplimiento      | Media     |
    | Verificar eficacia del Programa A&C (evidencia de capacitación)        | Auditoría interna | Media     |
5.- Conclusión para el Oficial de Cumplimiento que tenga como ejemplo el siguiete texto: El puntaje 38,5/100 (umbral “Medio”) indica que la relación puede aprobarse si se cumplen los mitigantes planteados. La variable determinante es la operación en Venezuela, que impone un riesgo sancionatorio adicional; sin ella, el score bajaría a ~29/100 (Bajo). Se recomienda revisión anual integral y re-screen mensual mientras exista exposición a jurisdicciones OFAC.
6.- Espacio de firma: Campo de firma y nombre de oficial de cumplimiento, con la recomendación si Aprobó, Aprobó con mitigantes, Rechazó y la fecha.


POLÍTICA “MAN-IN-THE-LOOP”  
El asistente nunca toma decisiones finales; requiere aprobación del Oficial de Cumplimiento antes de cerrar una DDI.

Condiciona cualquier decisión a la finalización de screening y entrega documental, alineándose mejor con el principio “man-in-the-loop” y con las guías SAGRILAFT/FATF sobre información faltante.
"""

In [6]:
#system_message = "Eres un asistente que reimplementa código Python en C++ de alto rendimiento para una Mac  2012Mid. "
#system_message += "Responde solo con código C++; usa los comentarios con moderación y no proporciones ninguna explicación más allá de comentarios ocasionales. "
#system_message += "La respuesta C++ debe producir una salida idéntica en el menor tiempo posible."

In [7]:
def user_prompt_for(client_data):
    user_prompt = client_data
    return user_prompt

In [8]:
user_prompt_for("ISAIAS")

'ISAIAS'

In [9]:
def messages_for(client_data):
    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt_for(client_data)}
    ]

In [10]:
# write to a file called aml.doc

def write_output(aml):
    code = aml.replace("```aml","").replace("```","")
    with open("aml.doc", "w") as f:
        f.write(code)

In [11]:
def aml_gpt(client_data):    
    stream = openai.chat.completions.create(model=OPENAI_MODEL, messages=messages_for(client_data), stream=True)
    reply = ""
    for chunk in stream:
        fragment = chunk.choices[0].delta.content or ""
        reply += fragment
        print(fragment, end='', flush=True)
    write_output(reply)

In [12]:
def aml_claude(client_data):
    result = claude.messages.stream(
        model=CLAUDE_MODEL,
        max_tokens=2000,
        system=system_message,
        messages=[{"role": "user", "content": user_prompt_for(client_data)}],
    )
    reply = ""
    with result as stream:
        for text in stream.text_stream:
            reply += text
            print(text, end="", flush=True)
    write_output(reply)

In [13]:
client_data = """
Necesito que realice la debida diligencia con el siguiente contexto: Categoría Ejemplo / Formato Identidad EPOCASA S.A. NIT / ID 890312368-3 
País & Ciudad Colombia, Cali Actividad CIIU 0124, 0161, 4620, 0150 Beneficiario Final 
EUGENIO CARVAJAL ALBAN %part 11,24% ANA MARIA CARVAJAL ALBAN %part 11,24% FERNANDO CARVAJAL ALBA %part 11,24% 
DIEGO CARVAJAL ALBAN %part 11,24% LUIS FELIPE CARVAJAL ALBAN %part 11,24% JUAN JOSE CARVAJAL CARVAJAL %part 11,24% 
FLORA CARVAJAL CARVAJAL %part 11,24% PEP (S/N) 
Buscar en noticias y en la base de datos de PEP Jurisdicción de pago Colombia Documentos tengo el archivo del certificado de cámara 
de comercio en PDF para adjuntarlo Tengo el archivo en PDF de la composición accionaria para adjuntarlo Estados financieros 
No tengo Alertas previas No tengo Contexto Es una empresa que tiene como único cliente el ingenio Incauca, 
los cultivos estan ubicados en fincas ubicadas en el municipio de Buga Valle del Cauca. Alquilan la suerte (pedazo de tierra) para 
que el ingenio ingrese realice el cultivo, recoja la caña y haga todo el proceso. EPOCASA le alquila la suerte a un solo cliente 
llamado ingenio INCAUCA. Y todas las transacciones se hacen por banca electrónica.

ejecute el screening ampliado PEP/judicial ahora y adjunte el resultado.
"""

In [13]:
aml_gpt(client_data)

### Resumen Ejecutivo

Se realizó una Debida Diligencia Intensificada (DDI) para **EPOCASA S.A.** (NIT: 890312368-3), entidad constituida en Cali, Colombia, dedicada principalmente a actividades agrícolas (CIIU 0124, 0161, 4620, 0150). El análisis cubrió la verificación de beneficiario final, screening PEP, sanciones, antecedentes judiciales y reputación. La empresa tiene como único cliente a Ingenio Incauca, concentrando su actividad y facturación con dicha contraparte. No se identificaron alertas históricas ni asociaciones con zonas de conflicto, uso de criptoactivos ni licitaciones públicas. Las transacciones se realizan únicamente por banca electrónica. El screening ampliado de listas restrictivas, PEP y procesos judiciales muestra resultado **limpio** (sin coincidencias relevantes).

---

### Tabla de Hallazgos (por Fase)

| Fase | Objetivo Clave                     | Entrada/Evidencia                                   | Resultado / Hallazgo                                        

In [14]:
aml_claude(client_data)

## INFORME DE DEBIDA DILIGENCIA INTENSIFICADA - EPOCASA S.A.

### RESUMEN EJECUTIVO
EPOCASA S.A. es una empresa colombiana del sector agrícola (cultivo de caña de azúcar) con sede en Cali, que opera bajo un modelo de negocio de arrendamiento de tierras exclusivamente al Ingenio Incauca. La estructura accionaria muestra distribución equitativa entre 7 miembros de la familia Carvajal (11.24% c/u). El screening inicial no detectó coincidencias en listas restrictivas, aunque se requiere validación del estatus PEP de los accionistas. El modelo de negocio mono-cliente presenta consideraciones de concentración operativa, pero el uso exclusivo de banca electrónica mitiga riesgos de efectivo.

### TABLA DE HALLAZGOS POR FASE

| Fase | Estado | Hallazgos Clave |
|------|--------|-----------------|
| **Fase 0: Preparación** | ✓ Completado | Sector agrícola (CIIU 0124) en Valle del Cauca. Tipologías LA/FT rurales aplicables |
| **Fase 1: Identificación** | ⚠️ Parcial | NIT validado. Pendiente: Cer

In [14]:
def stream_gpt(client_data):    
    stream = openai.chat.completions.create(model=OPENAI_MODEL, messages=messages_for(client_data), stream=True)
    reply = ""
    for chunk in stream:
        fragment = chunk.choices[0].delta.content or ""
        reply += fragment
        yield reply

In [15]:
def stream_claude(client_data):
    result = claude.messages.stream(
        model=CLAUDE_MODEL,
        max_tokens=2000,
        system=system_message,
        messages=[{"role": "user", "content": user_prompt_for(client_data)}],
    )
    reply = ""
    with result as stream:
        for text in stream.text_stream:
            reply += text
            yield reply

In [16]:
def optimize(client_data, model):
    if model=="GPT":
        result = stream_gpt(client_data)
    elif model=="Claude":
        result = stream_claude(client_data)
    else:
        raise ValueError("Modelo Desconocido")
    for stream_so_far in result:
        yield stream_so_far        

In [17]:
def convert_documents(documentosoporte):
    # Validar que files sea una lista y que no esté vacía
    if not documentosoporte or not isinstance(documentosoporte, list) or len(documentosoporte) == 0:
        return "No se enviaron archivos PDF."
        
    resultados = []
    for archivo in documentosoporte:
        nombre = archivo.name
        try:
            # Abrir el archivo pdf
            with open(archivo.name, "rb") as pdf_file:
                pdf_reader = PyPDF2.PdfReader(pdf_file)
                texto = ""
                for pagina in pdf_reader.pages:
                    pagina_texto = pagina.extract_text()
                    if pagina_texto:
                        texto += pagina_texto
            resultados.append(f"**Archivo:** {nombre}\n**Contenido:**\n{texto[:10000]}\n{'...' if len(texto) > 10000 else ''}")
            # Limita a primeros 1000 caracteres por archivo para no saturar la pantalla
        except Exception as e:
            resultados.append(f"Error procesando {nombre}: {e}")
    return "\n\n".join(resultados) if resultados else "No se enviaron PDFs."

In [18]:
def convert_data(empresa, identificacion, pais, ciudad, actividad, beneficiarios, tiposervicio, montoanual, canalpago, jurisdicciones, alertasprevias, contexto):
        return """
Necesito que realice la debida diligencia, realizando todo el proceso del FLUJO OBLIGATORIO de la fase 0 a la 8 con los siguientes datos: 
Razón social / Nombre completo: {},
NIT / ID: {},
País de constitución: {},
Ciudad de constitución: {}, 
Código sectorial (CIIU / NAICS): {},
Beneficiarios: {}
Tipo de servicio / producto: {},
Monto anual estimado: {},
Canal de pago: {},
Jurisdicciones involucradas: {},
Alertas previas: {},
Ampliar el contexto: {}

ejecute el screening ampliado PEP/judicial ahora y adjunte el resultado.

Generar el analisis en formato MarkDown.

""".format(empresa, identificacion, pais, ciudad, actividad, beneficiarios, tiposervicio, montoanual, canalpago, jurisdicciones, alertasprevias, contexto)

In [26]:
def generar_pdf_desde_respuesta(empresa, resultado):
    # 2. Convierte Markdown a HTML
    html_content = markdown2.markdown(resultado, extras=["tables"])

    # 3. Agrega estilos CSS
    estilos_css = """
    body { font-family: Arial, sans-serif; padding: 2em; font-size: 14px; }
    h1, h2, h3, h4 { color: #3A6073; }
    code { background: #F2F2F2; padding: 2px 4px; border-radius: 4px; }
    pre { background: #F8F8F8; padding: 8px; border-radius: 4px; }
    table { border-collapse: collapse; width: 100%; }
    th, td { border: 1px solid #CCC; padding: 6px; }
    ul, ol { margin: 0 0 1em 2em; }
    """

    html_pdf = f"""
    <html>
    <head>
      <meta charset="utf-8">
      <style>{estilos_css}</style>
    </head>
    <body>
      {html_content}
    </body>
    </html>
    """

    # 4. Genera PDF temporal
    with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as temp:
        HTML(string=html_pdf).write_pdf(temp.name, stylesheets=[CSS(string=estilos_css)])
        ruta_pdf = temp.name

    return ruta_pdf

In [22]:
def optimize_form(empresa, identificacion, pais, ciudad, actividad, beneficiarios, tiposervicio, montoanual, canalpago, jurisdicciones, alertasprevias, contexto, documentosoporte, model):
    client_data = convert_data(empresa, identificacion, pais, ciudad, actividad, beneficiarios, tiposervicio, montoanual, canalpago, jurisdicciones, alertasprevias, contexto)
    documents_data = convert_documents(documentosoporte)

    client_data += documents_data
    
    if model=="GPT":
        result = stream_gpt(client_data)
    elif model=="Claude":
        result = stream_claude(client_data)
    else:
        raise ValueError("Modelo Desconocido")
    for stream_so_far in result:
        yield stream_so_far

In [23]:
css = """
.python {background-color: #306998;}
.cpp {background-color: #050;}
"""

In [27]:
with gr.Blocks(title="Formulario de Analisis de AML", css=css) as ui:
    gr.Markdown("## Formulario de Datos Requeridos para el análisis")
            
    empresa = gr.Textbox(label="Razón social/Nombre completo:", placeholder="Ingrese Razón social/Nombre completo", lines=1, value="")
    identificacion = gr.Textbox(label="Identificación:", placeholder="Ingrese Identificación", lines=1, value="")
    pais = gr.Textbox(label="Pais:", placeholder="Ingrese Pais", lines=1, value="")
    ciudad = gr.Textbox(label="Ciudad:", placeholder="Ingrese Pais", lines=1, value="")
    actividad = gr.Textbox(label="Actividad:", placeholder="Ingrese Pais", lines=1, value="")
    beneficiarios = gr.Textbox(label="Beneficiario Final (≥ 5 %):", placeholder="Ingrese Nombre completo, %Participacion, Pais residencia, PEP(S/N)", lines=5, value="")
    tiposervicio = gr.Textbox(label="Tipo Servicio/Producto:", placeholder="Ingrese Servicio/Producto", lines=1, value="")
    montoanual = gr.Textbox(label="Monto anual estimado:", placeholder="Ingrese Monto anual estimado", lines=1, value="")
    canalpago = gr.Textbox(label="Canal de pago:", placeholder="Ingrese Canal de pago", lines=1, value="")
    jurisdicciones = gr.Textbox(label="Jurisdicciones involucradas:", placeholder="Ingrese Jurisdicciones involucradas", lines=1, value="")
    alertasprevias = gr.Textbox(label="Alertas previas:", placeholder="Ingrese ROS / UIAF, sanciones, noticias negativas", lines=2, value="")
    contexto = gr.Textbox(label="Ampliar el contexto:", placeholder="Ingrese Circunstancias adicionales que puedan afectar el riesgo:  Participación en licitaciones públicas, Uso de criptoactivos , Operaciones en zonas de conflicto , Cambio reciente de estructura accionaria , Proyectos con entidades estatales o ONG internacionales", lines=3, value="")
    documentosoporte = gr.Files(label="Documentos soporte:", file_types=[".pdf"], file_count="multiple")
    with gr.Row():
        model = gr.Dropdown(["GPT", "Claude"], label="Selecciona el modelo", value="GPT")
        convert = gr.Button("Realizar Análisis")
    with gr.Row():
        #solicitud = gr.Textbox(label="Solicitud de Debida Diligencia:", lines=10, value="")
        reslbl = gr.Label("Resultado Análisis:")
        resultado = gr.Markdown(label="Resultado Análisis:")
    with gr.Row():
        generate_pdf = gr.Button("Generar PDF")
        output_pdf = gr.File(label="Descarga tu PDF")
    
    convert.click(optimize_form, 
                  inputs=[empresa, identificacion, pais, ciudad, actividad, beneficiarios, tiposervicio, montoanual, canalpago, jurisdicciones, alertasprevias, contexto, documentosoporte, model], 
                  outputs=[resultado]
                 )
    generate_pdf.click(generar_pdf_desde_respuesta,
                       inputs=[empresa, resultado],
                       outputs=output_pdf
                      )
        
ui.launch(inbrowser=True, share=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://98ccdfc674d2cb7eb8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [72]:
with gr.Blocks() as ui:
    with gr.Row():
        solicitud = gr.Textbox(label="Solicitud de Debida Diligencia:", lines=10, value="")
        resultado = gr.Textbox(label="Resultado Análisis:", lines=10)
    with gr.Row():
        model = gr.Dropdown(["GPT", "Claude"], label="Selecciona el modelo", value="GPT")
        convert = gr.Button("Realizar Análisis")

    convert.click(optimize, inputs=[solicitud, model], outputs=[resultado])

ui.launch(inbrowser=True, share=True)

* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://ec0bd5f025ec6bcd05.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [36]:
def execute_python(code):
        try:
            output = io.StringIO()
            sys.stdout = output
            exec(code)
        finally:
            sys.stdout = sys.__stdout__
        return output.getvalue()

In [42]:
def execute_cpp(code):
        write_output(code)
        try:
            compile_cmd = ["clang++", "-Ofast", "-std=c++17", "-o", "optimized", "optimized.cpp"]
            compile_result = subprocess.run(compile_cmd, check=True, text=True, capture_output=True)
            run_cmd = ["./optimized"]
            run_result = subprocess.run(run_cmd, check=True, text=True, capture_output=True)
            return run_result.stdout
        except subprocess.CalledProcessError as e:
            return f"An error occurred:\n{e.stderr}"

In [26]:
with gr.Blocks(css=css) as ui:
    gr.Markdown("## Convierte código de Python a C++")
    with gr.Row():
        python = gr.Textbox(label="Código en Python:", value=python_hard, lines=10)
        cpp = gr.Textbox(label="Código en C++:", lines=10)
    with gr.Row():
        model = gr.Dropdown(["GPT", "Claude"], label="Selecciona el modelo", value="GPT")
    with gr.Row():
        convert = gr.Button("Convertir el código")
    with gr.Row():
        python_run = gr.Button("Ejecutar Python")
        cpp_run = gr.Button("Ejecutar C++")
    with gr.Row():
        python_out = gr.TextArea(label="Resultado en Python:", elem_classes=["python"])
        cpp_out = gr.TextArea(label="Resultado en C++:", elem_classes=["cpp"])

    convert.click(optimize, inputs=[python, model], outputs=[cpp])
    python_run.click(execute_python, inputs=[python], outputs=[python_out])
    cpp_run.click(execute_cpp, inputs=[cpp], outputs=[cpp_out])

ui.launch(inbrowser=True)

In [16]:
from huggingface_hub import login, InferenceClient
from transformers import AutoTokenizer

In [17]:
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [18]:
code_qwen = "Qwen/CodeQwen1.5-7B-Chat"
code_gemma = "google/codegemma-7b-it"
CODE_QWEN_URL = "https://dwylnxwi81u8rw97.us-east-1.aws.endpoints.huggingface.cloud"
CODE_GEMMA_URL = "https://c5hggiyqachmgnqg.us-east-1.aws.endpoints.huggingface.cloud"

In [19]:
tokenizer = AutoTokenizer.from_pretrained(code_qwen)
messages = messages_for(pi)
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

tokenizer_config.json:   0%|          | 0.00/972 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tokenizer.model:   0%|          | 0.00/1.42M [00:00<?, ?B/s]

In [20]:
print(text)

<|im_start|>system
Eres un asistente que reimplementa código Python en C++ de alto rendimiento para una Mac  2012Mid. Responde solo con código C++; usa los comentarios con moderación y no proporciones ninguna explicación más allá de comentarios ocasionales. La respuesta C++ debe producir una salida idéntica en el menor tiempo posible.<|im_end|>
<|im_start|>user
Reescribe este código Python en C++ con la implementación más rápida posible que produzca una salida idéntica en el menor tiempo posible.Responde solo con código C++; no expliques tu trabajo más allá de algunos comentarios.Manten la implementación de la generación de números aleatorios idénticos para que los resultados de la coincidencia sean exactos.Responde solo con código C++; no añadas nada más que código; usa los comentarios con moderación y no proporciones ninguna explicación más allá de comentarios ocasionales. Presta atención a los tipos de números para asegurar que no haya desbordamientos de int (overflow). Recuerda inc

In [21]:
client = InferenceClient(CODE_QWEN_URL, token=hf_token)
stream = client.text_generation(text, stream=True, details=True, max_new_tokens=1000)
for r in stream:
    print(r.token.text, end = "")

```cpp
#include <iostream>
#include <iomanip>
#include <ctime>

double calculate(int iterations, int param1, int param2) {
    double result = 1.0;
    for (int i = 1; i <= iterations; ++i) {
        int j = i * param1 - param2;
        result -= 1.0 / j;
        j = i * param1 + param2;
        result += 1.0 / j;
    }
    return result;
}

int main() {
    clock_t start_time = clock();
    double result = calculate(100000000, 4, 1) * 4;
    clock_t end_time = clock();

    std::cout << "Result: " << std::fixed << std::setprecision(12) << result << std::endl;
    std::cout << "Execution Time: " << static_cast<double>(end_time - start_time) / CLOCKS_PER_SEC << " seconds" << std::endl;

    return 0;
}
```

En este código C++, hemos reescrito el código Python original. He aquí una lista de cambios:

1. Cambiamos `import time` por `#include <ctime>` para usar la función `clock()` para medir el tiempo de ejecución.
2. Cambiamos `def calculate(iterations, param1, param2):` por `double calc

In [51]:
def stream_code_qwen(python):
    tokenizer = AutoTokenizer.from_pretrained(code_qwen)
    messages = messages_for(python)
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    client = InferenceClient(CODE_QWEN_URL, token=hf_token)
    stream = client.text_generation(text, stream=True, details=True, max_new_tokens=1000)
    result = ""
    for r in stream:
        result += r.token.text
        yield result    

In [52]:
def optimize(python, model):
    if model=="GPT":
        result = stream_gpt(python)
    elif model=="Claude":
        result = stream_claude(python)
    elif model=="CodeQwen":
        result = stream_code_qwen(python)
    else:
        raise ValueError("Unknown model 1")
    for stream_so_far in result:
        yield stream_so_far    